# Module 10 • Advanced Applications
# Lesson 58 • Advanced Text Summarization — Extractive, Abstractive, Long-Document, and Faithfulness-Aware Summarization

**Course:** Natural Language Processing: From Foundations to Large Language Models  
**Author:** Eman Khater  
**Difficulty:** Advanced  
**Execution target:** CPU only

## Scope
This lesson builds a complete offline summarization workflow covering extractive and
abstractive concepts, sentence ranking, redundancy control, compression, long-document
chunking, faithfulness, hallucination risk, ROUGE-style evaluation, Arabic considerations,
and production monitoring.

## Learning Objectives
- distinguish extractive and abstractive summarization;
- rank sentences by importance;
- reduce redundancy;
- control summary length;
- summarize long documents hierarchically;
- explain faithfulness and factual consistency;
- compute ROUGE-style metrics;
- evaluate coverage and redundancy;
- detect unsupported summary statements;
- design Arabic-aware summarization.

## Table of Contents
1. Summarization Problem
2. Extractive vs Abstractive
3. Compression Ratio
4. Sentence Segmentation
5. TF-IDF Sentence Representation
6. Centroid Importance
7. Position Heuristic
8. Query-Focused Summarization
9. Redundancy Control
10. MMR
11. Source Document
12. Sentence Scoring
13. Extractive Summary
14. Length Control
15. Abstractive Concepts
16. Compression by Rewriting
17. Faithfulness
18. Hallucination
19. Source-Support Check
20. Coverage
21. Redundancy
22. Long-Document Summarization
23. Chunking
24. Chunk-Level Summaries
25. Hierarchical Merge
26. Map-Reduce Summarization
27. Sliding Windows
28. Context-Limit Trade-Offs
29. ROUGE-1
30. ROUGE-2
31. ROUGE-L
32. Evaluation Dataset
33. System Comparison
34. Faithfulness Evaluation
35. Error Taxonomy
36. Omission Errors
37. Distortion Errors
38. Unsupported Additions
39. Repetition
40. Multilingual Summarization
41. Arabic Summarization
42. Tashkeel Policy
43. Production Pipeline
44. Latency and Cost
45. Monitoring
46. Human Evaluation
47. Optional Pretrained Template
48. Reproducibility
49. Knowledge Check
50. Exercises
51. Summary and Next Lesson

# 1. Summarization Problem

A useful summary balances **coverage**, **compression**, **coherence**,
**non-redundancy**, and **faithfulness**.

In [ ]:
import platform
import re
import time
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

SEED = 42
np.random.seed(SEED)

pd.DataFrame(
    [
        ("Extractive", "select source sentences", "high source faithfulness"),
        ("Abstractive", "generate new wording", "flexible compression"),
        ("Query-focused", "summarize for a user need", "task relevance"),
        ("Long-document", "hierarchical/chunked processing", "context scalability"),
    ],
    columns=["Type", "Mechanism", "Advantage"],
)

# 2. Extractive vs Abstractive Summarization

Extractive systems copy source spans. Abstractive systems paraphrase and combine
information, which can improve readability but increase hallucination risk.

# 3. Compression Ratio

```text
compression_ratio = summary_length / source_length
```

In [ ]:
def compression_ratio(source, summary):
    source_tokens = source.split()
    return 0.0 if not source_tokens else len(summary.split()) / len(source_tokens)

# 4. Sentence Segmentation

In [ ]:
SENTENCE_PATTERN = re.compile(r"(?<=[.!?])\s+")

def split_sentences(text):
    return [s.strip() for s in SENTENCE_PATTERN.split(text.strip()) if s.strip()]

# 5. TF-IDF Sentence Representation

Sentence vectors provide a transparent representation for extractive ranking.

# 6. Centroid-Based Importance

A document centroid approximates its central semantic direction. Sentences near the
centroid tend to represent central content.

# 7. Position Heuristic

Position can help in news and technical writing, but it should be treated as a feature
rather than a universal rule.

# 8. Query-Focused Summarization

Query-focused summarization combines general sentence importance with relevance to the
user's information need.

# 9. Redundancy Control

Selecting top sentences independently can produce near-duplicates. Redundancy penalties
encourage broader coverage.

# 10. Maximal Marginal Relevance

```text
MMR = λ × relevance(candidate) - (1-λ) × similarity(candidate, selected)
```

# 11. Offline Source Document

In [ ]:
document = (
    "Machine translation converts text from one language into another. "
    "Statistical machine translation relied on phrase tables and language models. "
    "Neural machine translation replaced many hand-engineered components with end-to-end neural networks. "
    "Attention mechanisms improved encoder-decoder models by allowing the decoder to access all encoder states. "
    "Transformers removed recurrence and rely heavily on self-attention. "
    "Modern multilingual translation models can support many language directions in one checkpoint. "
    "Arabic translation is challenging because Arabic has rich morphology, attached clitics, and optional tashkeel. "
    "Evaluation should include more than one automatic metric. "
    "BLEU measures n-gram overlap, while chrF uses character n-grams and is useful for morphologically rich languages. "
    "Learned metrics such as COMET can provide additional semantic evaluation. "
    "Statistical confidence intervals and significance tests help determine whether observed system differences are reliable. "
    "Human error analysis remains necessary because automatic metrics do not explain every failure."
)

sentences = split_sentences(document)
pd.DataFrame({"sentence_id": range(len(sentences)), "sentence": sentences})

# 12. Sentence Scoring

In [ ]:
vectorizer = TfidfVectorizer(stop_words="english", ngram_range=(1, 2))
sentence_matrix = vectorizer.fit_transform(sentences)
centroid = np.asarray(sentence_matrix.mean(axis=0))
centroid_scores = cosine_similarity(sentence_matrix, centroid).ravel()
position_scores = np.array([1.0 / (1.0 + i) for i in range(len(sentences))])
importance_scores = 0.85 * centroid_scores + 0.15 * position_scores

pd.DataFrame({
    "sentence_id": range(len(sentences)),
    "importance": importance_scores,
    "sentence": sentences,
}).sort_values("importance", ascending=False).head()

# 13. Extractive Summary

In [ ]:
def extractive_summary(text, n_sentences=3, redundancy_weight=0.35):
    sentence_list = split_sentences(text)
    if len(sentence_list) <= n_sentences:
        return " ".join(sentence_list)

    local_vectorizer = TfidfVectorizer(stop_words="english", ngram_range=(1, 2))
    matrix = local_vectorizer.fit_transform(sentence_list)
    centroid = np.asarray(matrix.mean(axis=0))
    relevance = cosine_similarity(matrix, centroid).ravel()

    selected = []

    while len(selected) < n_sentences:
        best_index = None
        best_score = -np.inf

        for index in range(len(sentence_list)):
            if index in selected:
                continue

            redundancy = 0.0
            if selected:
                redundancy = float(
                    cosine_similarity(matrix[index], matrix[selected]).ravel().max()
                )

            score = relevance[index] - redundancy_weight * redundancy

            if score > best_score:
                best_score = score
                best_index = index

        selected.append(best_index)

    selected.sort()
    return " ".join(sentence_list[index] for index in selected)

summary = extractive_summary(document, n_sentences=4)

print(summary)
print("Compression ratio:", round(compression_ratio(document, summary), 3))

# 14. Length Control

In [ ]:
def trim_to_words(text, max_words):
    return " ".join(text.split()[:max_words])

trim_to_words(summary, 35)

# 15. Abstractive Summarization Concepts

Abstractive systems may merge, paraphrase, generalize, or compress multiple source
statements. Every generated claim should remain supported by the source.

# 16. Compression by Rewriting

In [ ]:
def simple_abstractive_compression(extractive_text):
    replacements = {
        "Machine translation": "MT",
        "machine translation": "MT",
        "automatic metrics": "automatic scores",
        "evaluation should include": "evaluation benefits from",
    }

    output = extractive_text
    for old, new in replacements.items():
        output = output.replace(old, new)
    return output

compressed_summary = simple_abstractive_compression(summary)
compressed_summary

# 17. Faithfulness

A faithful summary contains claims supported by the source. Faithfulness is different
from topical relevance.

# 18. Hallucination

Abstractive systems may introduce unsupported entities, altered numbers, reversed
relationships, invented causes, or unjustified certainty.

# 19. Source-Support Check

In [ ]:
def content_terms(text):
    return {
        token
        for token in re.findall(r"\b\w+\b", text.lower())
        if len(token) > 3
    }

def lexical_support_score(source, claim):
    source_terms = content_terms(source)
    claim_terms = content_terms(claim)
    if not claim_terms:
        return 1.0
    return len(source_terms & claim_terms) / len(claim_terms)

pd.Series({
    "supported": lexical_support_score(
        document,
        "Transformers rely heavily on self-attention."
    ),
    "unsupported": lexical_support_score(
        document,
        "Transformers were invented in 1995 for speech recognition."
    ),
})

This lexical support score is only a diagnostic. Semantic entailment or human review is
needed for stronger factual verification.

# 20. Coverage

In [ ]:
def sentence_coverage(source, summary_text, similarity_threshold=0.25):
    source_sentences = split_sentences(source)
    summary_sentences = split_sentences(summary_text)

    if not source_sentences or not summary_sentences:
        return 0.0

    combined = source_sentences + summary_sentences
    vectorizer = TfidfVectorizer(stop_words="english")
    matrix = vectorizer.fit_transform(combined)

    source_matrix = matrix[:len(source_sentences)]
    summary_matrix = matrix[len(source_sentences):]

    similarities = cosine_similarity(source_matrix, summary_matrix)
    covered = similarities.max(axis=1) >= similarity_threshold
    return float(covered.mean())

sentence_coverage(document, summary)

# 21. Redundancy

In [ ]:
def summary_redundancy(summary_text):
    sentence_list = split_sentences(summary_text)

    if len(sentence_list) < 2:
        return 0.0

    vectorizer = TfidfVectorizer(stop_words="english")
    matrix = vectorizer.fit_transform(sentence_list)
    sim = cosine_similarity(matrix)
    values = sim[np.triu_indices(len(sentence_list), k=1)]
    return float(values.mean())

summary_redundancy(summary)

# 22. Long-Document Summarization

Long documents may exceed model context limits. Common solutions include chunking,
section-based processing, map-reduce summarization, hierarchical summarization, and
retrieval-guided selection.

# 23. Chunking

In [ ]:
long_document = " ".join(
    [
        document,
        document.replace("Machine translation", "Natural language processing"),
        document.replace("translation", "summarization"),
    ]
)

def chunk_sentences(text, sentences_per_chunk=5):
    sentence_list = split_sentences(text)
    return [
        sentence_list[i:i+sentences_per_chunk]
        for i in range(0, len(sentence_list), sentences_per_chunk)
    ]

chunks = chunk_sentences(long_document, 5)
len(chunks)

# 24. Chunk-Level Summaries

In [ ]:
chunk_summaries = [
    extractive_summary(" ".join(chunk), n_sentences=2)
    for chunk in chunks
]

pd.DataFrame({
    "chunk_id": range(len(chunk_summaries)),
    "chunk_summary": chunk_summaries,
})

# 25. Hierarchical Merge

In [ ]:
hierarchical_summary = extractive_summary(
    " ".join(chunk_summaries),
    n_sentences=4,
)

hierarchical_summary

# 26. Map-Reduce Summarization

```text
document → chunks → chunk summaries → merged summary
```

This is a practical strategy for documents longer than a model's input window.

# 27. Sliding Windows

Overlapping windows reduce boundary losses but increase computation and duplicate
content.

# 28. Context-Limit Trade-Offs

Larger chunks preserve more local context but cost more memory and latency. Smaller
chunks are cheaper but may fragment important dependencies.

# 29. ROUGE-1

In [ ]:
def rouge_n_f1(reference, candidate, n=1):
    ref_tokens = reference.lower().split()
    cand_tokens = candidate.lower().split()

    ref_counts = Counter(
        tuple(ref_tokens[i:i+n])
        for i in range(len(ref_tokens)-n+1)
    )

    cand_counts = Counter(
        tuple(cand_tokens[i:i+n])
        for i in range(len(cand_tokens)-n+1)
    )

    overlap = sum((ref_counts & cand_counts).values())
    ref_total = sum(ref_counts.values())
    cand_total = sum(cand_counts.values())

    if ref_total == 0 or cand_total == 0:
        return 0.0

    precision = overlap / cand_total
    recall = overlap / ref_total

    if precision + recall == 0:
        return 0.0

    return 2 * precision * recall / (precision + recall)

# 30. ROUGE-2

In [ ]:
reference_summary = (
    "Transformers use self-attention for modern machine translation. "
    "Arabic translation is affected by morphology and tashkeel. "
    "Evaluation should combine automatic metrics with human analysis."
)

pd.Series({
    "ROUGE-1 F1": rouge_n_f1(reference_summary, summary, 1),
    "ROUGE-2 F1": rouge_n_f1(reference_summary, summary, 2),
})

# 31. ROUGE-L

In [ ]:
def lcs_length(a, b):
    dp = np.zeros((len(a)+1, len(b)+1), dtype=int)

    for i in range(1, len(a)+1):
        for j in range(1, len(b)+1):
            if a[i-1] == b[j-1]:
                dp[i, j] = dp[i-1, j-1] + 1
            else:
                dp[i, j] = max(dp[i-1, j], dp[i, j-1])

    return int(dp[-1, -1])

def rouge_l_f1(reference, candidate):
    ref = reference.lower().split()
    cand = candidate.lower().split()

    if not ref or not cand:
        return 0.0

    lcs = lcs_length(ref, cand)
    precision = lcs / len(cand)
    recall = lcs / len(ref)

    if precision + recall == 0:
        return 0.0

    return 2 * precision * recall / (precision + recall)

rouge_l_f1(reference_summary, summary)

# 32. Evaluation Dataset

In [ ]:
evaluation_cases = [
    {
        "document": (
            "Neural networks learn representations from data. "
            "Recurrent networks process sequences step by step. "
            "Transformers use attention and allow more parallel computation."
        ),
        "reference": (
            "Transformers use attention and enable more parallel sequence processing."
        ),
    },
    {
        "document": (
            "Arabic has rich morphology and attached clitics. "
            "Tashkeel can distinguish different readings. "
            "Tokenization choices affect Arabic NLP systems."
        ),
        "reference": (
            "Arabic NLP is affected by morphology, clitics, tashkeel, and tokenization."
        ),
    },
    {
        "document": (
            "Retrieval systems rank documents for user queries. "
            "BM25 is lexical while dense retrieval uses vectors. "
            "Hybrid retrieval combines both signals."
        ),
        "reference": (
            "Hybrid retrieval combines lexical and vector-based search signals."
        ),
    },
]

len(evaluation_cases)

# 33. System Comparison

In [ ]:
evaluation_rows = []

for case_id, case in enumerate(evaluation_cases):
    candidate = extractive_summary(case["document"], n_sentences=1)

    evaluation_rows.append({
        "case_id": case_id,
        "candidate": candidate,
        "ROUGE-1": rouge_n_f1(case["reference"], candidate, 1),
        "ROUGE-2": rouge_n_f1(case["reference"], candidate, 2),
        "ROUGE-L": rouge_l_f1(case["reference"], candidate),
        "compression": compression_ratio(case["document"], candidate),
    })

evaluation_frame = pd.DataFrame(evaluation_rows)
evaluation_frame

# 34. Faithfulness Evaluation

In [ ]:
faithfulness_rows = []

for case_id, case in enumerate(evaluation_cases):
    candidate = evaluation_frame.loc[
        evaluation_frame["case_id"] == case_id,
        "candidate",
    ].iloc[0]

    faithfulness_rows.append({
        "case_id": case_id,
        "lexical_support": lexical_support_score(case["document"], candidate),
    })

pd.DataFrame(faithfulness_rows)

# 35. Error Taxonomy

In [ ]:
pd.DataFrame(
    [
        ("Omission", "important information missing"),
        ("Unsupported addition", "summary adds unsupported information"),
        ("Distortion", "meaning or relationship changed"),
        ("Entity error", "wrong entity, number, or attribute"),
        ("Redundancy", "same content repeated"),
        ("Over-compression", "too much important content removed"),
        ("Under-compression", "summary remains unnecessarily long"),
        ("Coherence", "summary statements do not connect well"),
    ],
    columns=["Error type", "Description"],
)

# 36. Omission Errors

Summaries must omit information. The relevant question is whether the omitted content
was essential to the user's goal.

# 37. Distortion Errors

A summary may use source vocabulary yet reverse causality, alter certainty, or merge
facts incorrectly. Overlap metrics cannot reliably detect such errors.

# 38. Unsupported Additions

Abstractive systems benefit from claim extraction, entailment checks, source links,
citations, and human review when factual risk is high.

# 39. Repetition

Repetition wastes the summary budget and can exaggerate one topic. Redundancy penalties
and MMR-style selection reduce this problem.

# 40. Multilingual Summarization

Summarization can be monolingual or cross-lingual. Cross-lingual summarization combines
translation and summarization challenges.

# 41. Arabic Summarization

Arabic summarization must consider morphology, clitics, orthographic variation,
sentence segmentation, named entities, optional tashkeel, and dialect variation.

# 42. Tashkeel Policy

In [ ]:
ARABIC_DIACRITICS = set(
    "\u064b\u064c\u064d\u064e\u064f\u0650\u0651\u0652"
)

def strip_tashkeel(text):
    return "".join(
        character
        for character in text
        if character not in ARABIC_DIACRITICS
    )

arabic_source = (
    "يُسَاعِدُ التَّلْخِيصُ عَلَى تَقْلِيلِ طُولِ النَّصِّ مَعَ الحِفَاظِ عَلَى المَعْلُومَاتِ المُهِمَّةِ."
)

pd.Series({
    "original": arabic_source,
    "diagnostic_without_tashkeel": strip_tashkeel(arabic_source),
})

For fully vocalized Arabic summarization, preserve tashkeel in the source, output, and
primary evaluation. A stripped version may be used only as a secondary diagnostic.

# 43. Production Pipeline

```text
document
  ↓
parsing / segmentation
  ↓
importance or retrieval stage
  ↓
summarization model
  ↓
faithfulness / support checks
  ↓
length and formatting rules
  ↓
summary + provenance
```

# 44. Latency and Cost

In [ ]:
def time_function(fn, *args, repeats=50, **kwargs):
    values = []

    for _ in range(repeats):
        start = time.perf_counter()
        fn(*args, **kwargs)
        values.append((time.perf_counter() - start) * 1000)

    return {
        "mean_ms": float(np.mean(values)),
        "p95_ms": float(np.percentile(values, 95)),
    }

time_function(
    extractive_summary,
    document,
    n_sentences=4,
)

# 45. Monitoring

Monitor input length, output length, compression ratio, latency, empty outputs,
repetition, unsupported claims, user corrections, language distribution, and model
version drift.

# 46. Human Evaluation

Human evaluation can score informativeness, relevance, coherence, fluency, faithfulness,
and usefulness. For high-stakes domains, factual verification matters more than style.

# 47. Optional Pretrained Summarization Template

This section is intentionally non-executable because it requires an external checkpoint.

```python
from transformers import pipeline

summarizer = pipeline(
    "summarization",
    model="your-seq2seq-summarization-model",
    device=-1,
)

summary = summarizer(
    document,
    max_new_tokens=120,
    min_new_tokens=30,
    do_sample=False,
)
```

Long documents should be chunked or processed hierarchically rather than blindly
truncated.

# 48. Reproducibility

In [ ]:
pd.Series(
    {
        "module": "Module 10 • Advanced Applications",
        "lesson": "Lesson 58 • Advanced Text Summarization",
        "source_sentences": len(sentences),
        "extractive_method": "TF-IDF centroid + redundancy penalty",
        "evaluation_cases": len(evaluation_cases),
        "seed": SEED,
        "offline_execution": True,
        "python": platform.python_version(),
    },
    name="Lesson 58 experiment",
)

# 49. Knowledge Check

1. What is the difference between extractive and abstractive summarization?
2. What does compression ratio measure?
3. Why can centroid similarity help sentence selection?
4. Why is redundancy control necessary?
5. What does MMR balance?
6. Why is faithfulness different from relevance?
7. What is a hallucinated summary claim?
8. Why is long-document summarization often hierarchical?
9. What does ROUGE-1 measure?
10. What does ROUGE-L use?
11. Why can ROUGE miss factual distortion?
12. What is cross-lingual summarization?
13. Why must fully vocalized Arabic preserve tashkeel?
14. What should a production summarization service monitor?
15. Why is human evaluation still important?

# 50. Exercises

1. Add a TextRank-style sentence graph.
2. Compare centroid ranking with position-only ranking.
3. Implement query-focused summarization.
4. Tune the redundancy penalty.
5. Compare several compression ratios.
6. Implement overlapping long-document windows.
7. Add ROUGE precision and recall separately.
8. Build a manual faithfulness annotation sheet.
9. Add fully vocalized Arabic documents.
10. Compare extractive and pretrained abstractive summaries on the same corpus.

## Challenge Exercises

1. Add a local pretrained summarization model.
2. Implement long-document map-reduce summarization.
3. Add entailment-based faithfulness checking.
4. Build citation-linked summaries with supporting source spans.
5. Compare summarization quality before and after retrieval-guided source selection.

# 51. Summary and Next Lesson

In this lesson:

- extractive and abstractive summarization were distinguished;
- sentence importance was estimated with TF-IDF and centroid similarity;
- redundancy-aware extraction was implemented;
- compression ratio and length control were measured;
- long-document chunking and hierarchical map-reduce summarization were built;
- faithfulness, hallucination, coverage, and redundancy were analyzed;
- ROUGE-1, ROUGE-2, and ROUGE-L-style metrics were implemented;
- multilingual and Arabic/tashkeel considerations were included;
- production latency, monitoring, and human evaluation were connected to system design.

## Next Lesson

**Lesson 59: Advanced Question Answering — Extractive, Generative, Retrieval-Augmented,
and Evidence-Grounded QA**

# References

- Nenkova, A. and McKeown, K. work on automatic summarization.
- Mihalcea, R. and Tarau, P. work on TextRank.
- Lin, C.-Y. work on ROUGE.
- See, A. et al. work on pointer-generator summarization.
- Lewis, M. et al. work on BART.
- Zhang, J. et al. work on PEGASUS.
- Research on factual consistency and faithfulness in neural summarization.